# Download Data
In this script, we view the available sites to do some modeling and download some data. In this notebook we do the following:

1. Define some solutes that we might be interested in modeling (+ some other data)
2. Query the USGS database to find what rivers have the data we are looking for
3. Download the streamflow and river chemistry data we are interested in
4. Get a polygon of the catchment we are interested in
5. Download precipitation + temperature data for the site we are interested in
6. Plot the downloaded data

## Get water quality data

In [112]:
import os
import dataretrieval.wqp as wqp
import dataretrieval.waterdata as waterdata
import pandas as pd
from pandas import DataFrame, Series
from geopandas import GeoDataFrame

data_dir: str = "../data"
if "snakemake" in globals():
    data_dir = "./data"

output_dir = os.path.join(data_dir, "raw")

In [2]:
doc_df, _ = wqp.get_results(
    pCode="00681",
    providers="NWIS",
    dataProfile="narrowResult",
)

/home/andrew/miniforge3/envs/potions-tutorial/lib/python3.14/site-packages/dataretrieval/wqp.py:664: DataCurrencyWarning: This function call will return the legacy WQX format, which means USGS data have not been updated since March 2024. Please review the dataretrieval-python documentation for more information on updated WQX3.0 profiles. Setting `legacy=False` will remove this warning.
  _warn_legacy_use()


In [3]:
relevant_cols: list[str] = [
    "ActivityStartDate",
    "ActivityStartTime/Time",
    "MonitoringLocationIdentifier",
    "ResultMeasureValue",
    "ResultMeasure/MeasureUnitCode",
    "ActivityStartDateTime",
]
sub_doc_df: DataFrame = doc_df[relevant_cols]

### Select a relevant site

In [7]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

sub_doc_df.to_csv(os.path.join(output_dir, "full_wq_data.csv"))

In [11]:
counts = sub_doc_df["MonitoringLocationIdentifier"].value_counts()
counts.nlargest(10)

MonitoringLocationIdentifier
USGS-01434025           3021
USGS-401733105392404    2007
USGS-01434021           1863
USGS-0143400680         1854
USGS-04253296           1659
USGS-01435000           1381
USGS-01364959           1327
USGS-04253295           1318
USGS-04253294           1281
USGS-01576787           1209
Name: count, dtype: int64

In [60]:
top_site: str = counts.index[0]  # type: ignore

In [61]:
site_df = sub_doc_df.loc[sub_doc_df["MonitoringLocationIdentifier"] == top_site]
site_df.index = pd.to_datetime(site_df.ActivityStartDateTime)
daily_site_doc: Series = (
    site_df["ResultMeasureValue"].rename("doc_mgl").resample("D").mean()
)  # Daily DOC in mg/L
daily_site_doc.index = daily_site_doc.index.tz_localize(None)

In [69]:
start_wy: int = 2010
end_wy: int = 2015
sim_start: pd.Timestamp = pd.Timestamp(year=start_wy - 1, month=10, day=1)
sim_end: pd.Timestamp = pd.Timestamp(year=end_wy, month=9, day=30)
meas_doc_mgl: Series = daily_site_doc[sim_start:sim_end]
meas_doc_molar: Series = (meas_doc_mgl / 12 / 1000).rename("doc_molar")

In [65]:
q_cfs_df, _meta = waterdata.get_daily(top_site, time=f"{str(sim_start)}/{str(sim_end)}")
q_cfs_df = q_cfs_df.loc[q_cfs_df.parameter_code == "00060"]
print(f"Number of observations: {len(q_cfs_df)}")

Retrieving: daily · 1 page · 2,191 rows


Number of observations: 2191


In [70]:
q_cfs_df.index = pd.to_datetime(q_cfs_df["time"])
q_meas_cfs: Series = q_cfs_df["value"]
q_meas_cms: Series = q_meas_cfs / 35.3147  # Convert to units of cubic meters per second

## Get the watershed boundary for a site

In [71]:
from pynhd import NLDI
import pygridmet as gridmet

In [113]:
forcing_start = (sim_start.date() - pd.Timedelta(days=365)).strftime("%Y-%m-%d")
forcing_end = sim_end.date().strftime("%Y-%m-%d")
dates = forcing_start, forcing_end

In [114]:
basin_gdf = NLDI().get_basins(top_site)
area_meters = basin_gdf.to_crs(epsg=5070).area.iloc[0]
q_meas_mms = q_meas_cms / area_meters * 1000
q_meas_mmd = q_meas_mms * 86_400

In [115]:
cq_df: DataFrame = pd.concat(
    [q_meas_mmd.rename("q_mmd"), meas_doc_molar.rename("doc_mol_l")], axis=1
)

In [116]:
ds = gridmet.get_bygeom(
    basin_gdf.geometry.iloc[0], dates, variables=["pr", "tmmn", "tmmx", "pet"]
)

/home/andrew/miniforge3/envs/potions-tutorial/lib/python3.14/site-packages/pygridmet/pygridmet.py:161: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'day' ('day',) The recommendation is to set join explicitly for this case.
  clm = xr.merge(_open_dataset(f) for f in clm_all_files).astype("f4")
/home/andrew/miniforge3/envs/potions-tutorial/lib/python3.14/site-packages/pygridmet/pygridmet.py:161: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set comp

In [117]:
daily = ds.mean(dim=("lat", "lon")).to_dataframe()
daily["temp"] = (daily["tmmn"] + daily["tmmx"]) / 2 - 273.15
daily = daily[["pr", "temp", "pet"]].rename(
    columns={
        "pr": "ppt",
        "temp": "temp",
        "pet": "pet",
    }
)

In [120]:
# Now, save the outputs
model_inputs_dir: str = os.path.join(data_dir, "model_inputs")
if not os.path.exists(model_inputs_dir):
    os.makedirs(model_inputs_dir)

daily.to_csv(os.path.join(model_inputs_dir, "forcing.csv"))
cq_df.to_csv(os.path.join(model_inputs_dir, "measured_data.csv"))